# Tablas Comparativas de Métricas XAI

Una tabla por corte @k con columnas **AggDiv · IXD · MIL · ECS** y una fila por algoritmo.

In [ ]:
import os, glob
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
MODO     = 'semi'   # 'muestra' | 'semi' | 'completo'
BASE_DIR = os.path.join('..', '..', 'output', f'metricas_evaluacion_{MODO}')

METRICAS = ['AggDiv', 'IXD', 'MIL', 'ECS']

if not os.path.exists(BASE_DIR):
    print(f'⚠️  Directorio no encontrado: {BASE_DIR}')
else:
    print(f'✅ {len(glob.glob(os.path.join(BASE_DIR, "*.csv")))} CSVs encontrados')

In [ ]:
# ── Carga y agregación ────────────────────────────────────────────────────────
def cargar_metricas(base_dir):
    registros = []
    for path in sorted(glob.glob(os.path.join(base_dir, '*.csv'))):
        basename = os.path.basename(path)
        metrica = next((m for m in METRICAS if f'_{m}_' in basename or f'_{m}.' in basename), None)
        if metrica is None:
            continue
        df = pd.read_csv(path)
        algoritmo = df['algoritmo'].iloc[0] if 'algoritmo' in df.columns else None
        if algoritmo is None:
            continue

        def _mean(col):
            return df[col].dropna().mean() if col in df.columns else np.nan

        if metrica == 'ECS':
            # ECS: una fila por hotel → agregamos por media
            registros.append(dict(
                algoritmo=algoritmo, metrica=metrica,
                **{f'v{k}': _mean(f'{metrica}{k}') for k in ['', '@1', '@3', '@5']}
            ))
        else:
            # AggDiv, IXD (por usuario) o MIL (fila única)
            for _, row in df.iterrows():
                alg = row.get('algoritmo', algoritmo)
                registros.append(dict(
                    algoritmo=alg, metrica=metrica,
                    **{f'v{k}': row.get(f'{metrica}{k}', np.nan) for k in ['', '@1', '@3', '@5']}
                ))
    return pd.DataFrame(registros)


df_raw = cargar_metricas(BASE_DIR)
print(f'Registros: {len(df_raw)} | Métricas: {df_raw["metrica"].unique().tolist()}')

In [ ]:
# ── Construcción de tablas por @k ─────────────────────────────────────────────
def tabla_para_k(df_raw, sufijo):
    col_val = f'v{sufijo}'
    filas = []
    for alg in sorted(df_raw['algoritmo'].unique()):
        fila = {'Algoritmo': alg}
        for m in METRICAS:
            sub = df_raw[(df_raw['algoritmo'] == alg) & (df_raw['metrica'] == m)]
            fila[m] = round(sub[col_val].mean(), 4) if not sub.empty else np.nan
        filas.append(fila)
    return pd.DataFrame(filas).set_index('Algoritmo')


tablas = {
    'Global (@todos)': tabla_para_k(df_raw, ''),
    '@1':              tabla_para_k(df_raw, '@1'),
    '@3':              tabla_para_k(df_raw, '@3'),
    '@5':              tabla_para_k(df_raw, '@5'),
}
print('Algoritmos:', list(tablas['@1'].index))

In [ ]:
# ── Visualización minimalista ─────────────────────────────────────────────────
#
# AggDiv, IXD, MIL → verde oscuro = mejor (más alto)
# ECS              → naranja/rojo  = más consistente (más alto = menos personalizado)

CMAPS = {'AggDiv': 'YlGn', 'IXD': 'YlGn', 'MIL': 'YlGn', 'ECS': 'YlOrRd'}
DESC  = {
    'AggDiv': '↑ más diverso',
    'IXD':    '↑ más diverso',
    'MIL':    '↑ más personalizado',
    'ECS':    '↑ más consistente',
}

TABLA_STYLES = [
    {'selector': 'table',
     'props': [('border-collapse', 'collapse'), ('font-family', 'Inter, sans-serif'),
               ('font-size', '13px'), ('width', 'auto')]},
    {'selector': 'caption',
     'props': [('font-size', '14px'), ('font-weight', '600'), ('text-align', 'left'),
               ('padding-bottom', '8px'), ('color', '#222')]},
    {'selector': 'th',
     'props': [('background', '#1a1a2e'), ('color', '#fff'), ('padding', '8px 16px'),
               ('text-align', 'center'), ('font-weight', '500'), ('letter-spacing', '0.03em')]},
    {'selector': 'th.row_heading',
     'props': [('text-align', 'left'), ('min-width', '160px')]},
    {'selector': 'td',
     'props': [('padding', '6px 16px'), ('text-align', 'center'), ('border-bottom', '1px solid #f0f0f0')]},
    {'selector': 'tr:hover td',
     'props': [('filter', 'brightness(0.93)')]},
]


def mostrar_tabla(tabla, titulo):
    cols = [m for m in METRICAS if m in tabla.columns]
    styler = (tabla[cols]
              .style
              .set_caption(titulo)
              .format('{:.4f}', na_rep='—')
              .set_table_styles(TABLA_STYLES))
    for col in cols:
        styler = styler.background_gradient(cmap=CMAPS[col], subset=[col], axis=0)
    display(styler)


for titulo, tabla in tablas.items():
    mostrar_tabla(tabla, f'Métricas XAI — {titulo}')
    print()   # espacio entre tablas

In [ ]:
# ── Exportar ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join('..', '..', 'output', 'visualizacion_tablas')
os.makedirs(OUTPUT_DIR, exist_ok=True)

nombres = ['tabla_metricas_global.csv', 'tabla_metricas_at1.csv',
           'tabla_metricas_at3.csv',    'tabla_metricas_at5.csv']

for nombre, tabla in zip(nombres, tablas.values()):
    tabla.to_csv(os.path.join(OUTPUT_DIR, nombre))

print(f'✅ Tablas exportadas a {OUTPUT_DIR}')